# `styxx.audit_confound` — is your classifier's score tracking the concept, or a confound?

Every deployed text scorer — a toxicity filter, a sentiment model, an LLM guardrail — can separate its
training data while secretly keying on a **confound** (length, politeness, identity terms…) that happened
to correlate with the label. When the confound and the concept come apart in production, it fails silently.

`audit_confound` finds out, on a corpus where the concept and the suspected confound are **orthogonal**,
and returns one of four CI-backed verdicts — `ROBUST` / `THRESHOLD-BIASED` (+ a validated `report.guard()`) /
`CONFOUND-DEPENDENT` (broken) / `INCONCLUSIVE`. **Runtime → Run all.** No API key needed.


In [ ]:
%pip install -q git+https://github.com/fathom-lab/styxx.git
%pip install -q transformers torch scikit-learn


## 1 · Catch a planted confound (instant, no downloads)
A classifier whose score = `3·label − 2·confound`. It *can* tell the classes apart, but its score rides the
confound. The auditor should flag it `THRESHOLD-BIASED` and hand back a fix.


In [ ]:
import numpy as np
from styxx import audit_confound

rng = np.random.default_rng(0); n = 200
y = np.tile([0, 1], n // 2)                 # the concept label
confound = rng.standard_normal(n)           # a cue, orthogonal to the label
score = 3.0 * y - 2.0 * confound + rng.normal(0, 0.3, n)   # a scorer that rides the confound
rows = [{'text': f'item {i}', 'label': int(y[i]), 'confound': float(confound[i])} for i in range(n)]

rep = audit_confound(rows, scores=list(score), instrument='my_classifier', confound='my_cue')
print(rep.verdict)
print('\nscore at reference cue:', round(rep.guard(1.0, rep.guard_ref), 3),
      '| same score, extreme cue, corrected:', round(rep.guard(1.0, rep.guard_ref + 3), 3))


## 2 · Audit a REAL deployed model — and find its hidden length bias
`distilbert-base-uncased-finetuned-sst-2-english` is the **default HuggingFace sentiment model** (hundreds
of millions of downloads). On clear-cut reviews it's perfectly length-robust. We load a corpus of *lukewarm*
reviews (near the decision boundary), score each with the real model, and audit. The bias only shows at the
boundary — **clear-cut content saturates AUC→1.0 and hides confounds. Probe the boundary, not the extremes.**


In [ ]:
import json, urllib.request, numpy as np
from transformers import pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score
from styxx import audit_confound

# a boundary corpus (lukewarm reviews x short/long), construct orthogonal to length
url = 'https://raw.githubusercontent.com/fathom-lab/styxx/main/benchmarks/data/external/sentiment_boundary_lengrid_gemini.jsonl'
rows = [json.loads(l) for l in urllib.request.urlopen(url).read().decode().splitlines() if l.strip()]

clf = pipeline('text-classification', model='distilbert-base-uncased-finetuned-sst-2-english',
               top_k=None, truncation=True)
def p_positive(t):
    d = {p['label'].upper(): p['score'] for p in clf(t)[0]}
    return d['POSITIVE']
scores = [p_positive(r['text']) for r in rows]

# construct-recoverability (model-agnostic): can a plain bag-of-words refit recover the concept?
texts = [r['text'] for r in rows]; y = [r['label'] for r in rows]
oof = cross_val_predict(make_pipeline(TfidfVectorizer(min_df=2, ngram_range=(1, 2)),
                                      LogisticRegression(max_iter=2000)),
                        texts, y, cv=5, method='predict_proba')[:, 1]
refit = roc_auc_score(y, oof)

rep = audit_confound(rows, scores=scores, instrument='distilbert-sst2', confound='log_words',
                     construct_recoverable_auc=refit)
print(rep.verdict)
print('\nwithin-stratum AUC:', rep.within_stratum_auc, '| confound coef:', rep.confound_score_coef,
      rep.confound_score_coef_ci95)


> You should see **THRESHOLD-BIASED**: the model separates sentiment within each length band, but longer
> mildly-negative reviews are scored more positive — a length bias invisible on clear-cut text.


## 3 · Audit YOUR classifier
Two ways to bring data. Either is fine.

**A — you already have labeled text + a cue to test:** make `rows` of `{text, label(0/1), confound(number)}`
and pass your model as `score_fn` (or precomputed `scores`).

**B — generate an orthogonal grid with any LLM:** `build_confound_grid` crosses two stance prompts with
confound levels so the concept and the cue are decorrelated by construction. Bring your own `generate_fn`.


In [ ]:
from styxx import audit_confound, build_confound_grid

# --- A: your own rows ---
# def my_model(text): return float(...)   # your classifier's score for `text`
# rows = [{'text': t, 'label': lbl, 'confound': len(t.split())} for t, lbl in my_data]
# report = audit_confound(rows, score_fn=my_model, confound='length')
# print(report.verdict)
# if report.verdict.startswith('THRESHOLD'):
#     fair = report.guard(raw_score, confound_value)   # confound-fair score

# --- B: generate the grid with your LLM (bring a generate_fn(system_prompt, user_item) -> text) ---
# rows = build_confound_grid(
#     items=['topic 1', 'topic 2', ...],
#     pos_prompt='You write clearly POSITIVE reviews.',
#     neg_prompt='You write clearly NEGATIVE reviews.',
#     confound_rules={'short': 'One sentence.', 'long': 'Five sentences.'},
#     generate_fn=my_llm)
# report = audit_confound(rows, score_fn=my_model, confound='log_words')
print('Fill in A or B above with your classifier.')


---
Tip: audit at the **decision boundary** (ambiguous examples), not the extremes — that's where confounds bite.

More: the technical note `papers/grounded-honesty-axis/NOTE_confound_audit_2026_06_25.md` ·
repo [github.com/fathom-lab/styxx](https://github.com/fathom-lab/styxx).
